# Example 3 — Realistic basic state (NCEP JFM 1979 + MJO forcing)

Manuscript Sect. 4.3. The model is linearised about a **realistic observed basic state** — the JFM 1979 NCEP reanalysis — and driven by a time-varying **MJO phase-1** heating to produce an extratropical teleconnection response.

## 1. Set up the model (T42 L20, Isca/GFDL core)

In [ ]:
# --- locate the LBM_isca package and fetch the input data from Zenodo --------
import os, sys, types, urllib.request
import numpy as np

root = os.path.abspath("")
while root != os.path.dirname(root):
    if os.path.isdir(os.path.join(root, "lbm_isca")):
        break
    root = os.path.dirname(root)
sys.path.insert(0, root)

# Input data are archived on Zenodo (DOI 10.5281/zenodo.24086134) rather than
# committed to the repo. They are downloaded into data/ once, then cached.
DATA = os.path.join(root, "data")
ZENODO = "https://zenodo.org/records/24086134/files"   # data DOI 10.5281/zenodo.24086134

def ensure(fname):
    """Return the local path to a data file, fetching it from Zenodo if absent."""
    path = os.path.join(DATA, fname)
    if not os.path.exists(path):
        os.makedirs(DATA, exist_ok=True)
        url = f"{ZENODO}/{fname}?download=1"
        print(f"downloading {fname} from Zenodo ...")
        urllib.request.urlretrieve(url, path)
    return path

for _f in ("bs.1979.bin", "frc.phase1.bin"):
    print(f"  {_f}: {ensure(_f)}")

from lbm_isca import Config, LBM, Spectral, VerticalGrid, asnumpy, gpu_available
from lbm_isca.forcing import SteadyForcing, MJOForcing
from lbm_isca.basicstate import load_basic_state

DEVICE = "gpu" if gpu_available() else "cpu"
sp = Spectral(device=DEVICE)
vg = VerticalGrid(device=DEVICE)
lon, lat, sig = asnumpy(sp.lon_deg), asnumpy(sp.lat_deg), asnumpy(vg.sig)
print(f"LBM_isca  T{sp.nmax}  {sp.nlon}x{sp.nlat}  L{vg.kmax}  backend={DEVICE}")

## 2. Basic state

The basic state is read from the binary file **`bs.1979.bin`** — NCEP pressure-level fields (JFM 1979), time-averaged over 3 months and interpolated to the model's sigma levels. It is archived on **Zenodo** ([DOI 10.5281/zenodo.24086134](https://doi.org/10.5281/zenodo.24086134)) and downloaded into `data/` by the cell above. This is the realistic counterpart of the two idealised states — *replace this one file* to run a different climatology.

> **This is the only block that changes between the three examples — swap it to run a different case.**

In [ ]:
# BASIC STATE — realistic NCEP reanalysis (JFM 1979 average)
bs_file = os.path.join(DATA, "bs.1979.bin")
bs = load_basic_state(bs_file, sp, vg, kmo=1, navg=3)   # first month, 3-mo avg
bs.summary()

## 3. Forcing

Time-varying **MJO phase-1** heating read from `frc.phase1.bin` (a realistic convective heating anomaly that marches day by day after a short spin-up).

In [ ]:
mjo = MJOForcing(os.path.join(DATA, "frc.phase1.bin"),
                 nlev=vg.kmax, ndays=73,
                 nt=int(round(86400.0 / 1200.0)), spinup=435)

## 4. Integrate

In [ ]:
model = LBM(bs, sp, vg, Config(ndays=35.0))
model.run(mjo_forcing=mjo, verbose=True)

## 5. Look at the response

In [ ]:
# --- quick look: upper-level (sigma ~ 0.175, ~200 hPa) streamfunction --------
import matplotlib.pyplot as plt

k = int(np.argmin(np.abs(sig - 0.175)))
psi = asnumpy(model.ga.psi[:, :, k]) if hasattr(model.ga, "psi") \
      else asnumpy(model.diagnostics()["psi"][:, :, k])

fig, ax = plt.subplots(figsize=(10, 4.5))
vmax = np.abs(psi).max() or 1.0
cf = ax.contourf(lon, lat, psi, levels=np.linspace(-vmax, vmax, 21),
                 cmap="RdBu_r", extend="both")
ax.contour(lon, lat, psi, levels=np.linspace(-vmax, vmax, 21),
           colors="k", linewidths=0.3)
ax.set_ylim(-80, 80); ax.set_xlabel("longitude"); ax.set_ylabel("latitude")
ax.set_title(r"Upper-level ($\sigma\approx0.175$) streamfunction $\psi'$"
             r" — stationary Rossby-wave train", loc="left")
fig.colorbar(cf, ax=ax, pad=0.02, fraction=0.025)
fig.tight_layout(); plt.show()

## 6. (optional) Save gridded output

In [ ]:
# --- (optional) write GrADS + netCDF output ----------------------------------
from lbm_isca.output import GradsWriter  # NetCDFWriter also available
os.makedirs(os.path.join(root, "out"), exist_ok=True)
# Re-run with a writer attached if you want the full gridded history on disk:
#   w = GradsWriter(os.path.join(root, "out", "realistic.1979.phase1.grd"), sp, vg)
#   model.run(mjo_forcing=mjo, writer=w)
print("final-state grid fields available as model.ga.u / .v / .t / .psi ...")